In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio
!{sys.executable} -m pip install torch_geometric
!{sys.executable} -m pip install ogb
!{sys.executable} -m pip install numpy pandas matplotlib scikit-learn networkx seaborn
!{sys.executable} -m pip install streamlit
!{sys.executable} -m pip install jupyter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.3/915.3 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 77.5 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

False CPU only


In [ ]:
import torch

# --- Tensor Creation ---
a = torch.tensor([1, 2, 3])
b = torch.zeros((3, 3))
c = torch.rand((3, 3))
d = torch.arange(0, 10, 2)
print(a, b, c, d)

# --- Indexing ---
m = torch.arange(12).reshape(3, 4)
print(m[0])        # first row
print(m[:, 1])      # second column
print(m[1:, 2:])    # slicing

# --- Reshaping ---
x = torch.arange(24)
print(x.view(2, 3, 4))
print(x.reshape(4, 6))
print(x.unsqueeze(0).shape, x.squeeze().shape)

# --- Matrix Multiplication ---
A = torch.rand(3, 4)
B = torch.rand(4, 5)
C = torch.matmul(A, B)        # or A @ B
print(C.shape)

# --- Broadcasting ---
v = torch.tensor([1, 2, 3])
M = torch.rand(3, 3)
print(M + v)   # v broadcast across rows

# --- Aggregation ---
t = torch.rand(4, 4)
print(t.sum(), t.mean(dim=0), t.max(), t.argmax())

# --- GPU ops (if available) ---
if torch.cuda.is_available():
    t_gpu = t.to("cuda")
    print(t_gpu.device)

tensor([1, 2, 3]) tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]) tensor([[0.9243, 0.6346, 0.7256],
        [0.1607, 0.2295, 0.2652],
        [0.9622, 0.6045, 0.9988]]) tensor([0, 2, 4, 6, 8])
tensor([0, 1, 2, 3])
tensor([1, 5, 9])
tensor([[ 6,  7],
        [10, 11]])
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17],
        [18, 19, 20, 21, 22, 23]])
torch.Size([1, 24]) torch.Size([24])
torch.Size([3, 5])
tensor([[1.4451, 2.8986, 3.2480],
        [1.5723, 2.0409, 3.5712],
        [1.4351, 2.2021, 3.2511]])
tensor(8.0491) tensor([0.4012, 0.6054, 0.4407, 0.5650]) tensor(0.9464) tensor(3)


In [ ]:
from ogb.nodeproppred import PygNodePropPredDataset
import networkx as nx
import matplotlib.pyplot as plt
import torch

dataset = PygNodePropPredDataset(name="ogbn-arxiv", root="data/")
data = dataset[0]
print(data)
print("Nodes:", data.num_nodes, "Edges:", data.num_edges, "Feature dim:", data.x.shape[1])

# --- Edge list ---
edge_index = data.edge_index
edge_list = edge_index.t().tolist()
print(edge_list[:5])

# --- Sample subgraph for visualization ---
import random
sample_nodes = random.sample(range(data.num_nodes), 50)
G = nx.Graph()
G.add_nodes_from(sample_nodes)
for u, v in edge_list:
    if u in sample_nodes and v in sample_nodes:
        G.add_edge(u, v)

plt.figure(figsize=(8, 8))
nx.draw(G, node_size=40, with_labels=False)
plt.title("Sample subgraph of OGBN-Arxiv")
plt.savefig("subgraph_sample.png")
plt.show()

# --- Node feature description ---
print("Feature vector stats:", data.x.mean().item(), data.x.std().item())

# --- Degree distribution ---
degrees = torch.bincount(edge_index[0])
plt.hist(degrees.numpy(), bins=50)
plt.xlabel("Degree"); plt.ylabel("Count"); plt.title("Degree Distribution")
plt.savefig("degree_distribution.png")
plt.show()

# --- Graph density ---
n = data.num_nodes
e = data.num_edges
density = e / (n * (n - 1))
print("Density:", density)

# --- Connected components (use undirected version for meaningful count) ---
Gfull = nx.from_edgelist(edge_list)
num_components = nx.number_connected_components(Gfull)
largest_cc = max(nx.connected_components(Gfull), key=len)
print("Number of components:", num_components, "Largest component size:", len(largest_cc))

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:08<00:00,  9.33it/s]


Extracting data/arxiv.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 5761.41it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 3421.13it/s]

Saving...



Done!


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch_geometric.data.data.DataEdgeAttr was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch_geometric.data.data.DataEdgeAttr])` or the `torch.serialization.safe_globals([torch_geometric.data.data.DataEdgeAttr])` context manager to allowlist this global if you trust this class/function.

Check the documentat